# 12 -- Confidence Taxonomy and Season Aggregation (Contact Luck v0.11)

Version 0.11 turns the Version 0.10 play-level attribution ledger into a
statistically honest season-level reporting layer, WITHOUT refitting,
recalibrating, or tuning ANY Version 0.2-0.10 model. This notebook inspects
the already-computed outputs of that pipeline:

  - `mlb_luck_score.scoring.component_confidence` -- four SEPARATE confidence
    taxonomies per play/component (never one collapsed number).
  - `mlb_luck_score.scoring.aggregate_attribution` -- additive batter-season
    totals that reconcile EXACTLY to Version 0.10's own accounting identity.
  - `mlb_luck_score.scoring.aggregation_uncertainty` -- game_pk-clustered
    percentile bootstrap intervals, conditional on the frozen models.
  - `mlb_luck_score.scoring.qualification` -- predetermined qualification
    thresholds, fixed BEFORE any real leaderboard was examined.
  - `mlb_luck_score.models.evaluate_aggregation_stability` -- descriptive
    2021-2024 reliability/sensitivity analysis (not an untouched validation).

Run `make run-season-aggregation` (Version 0.11) then `make evaluate-
aggregation-stability` to (re)generate the JSON reports this notebook reads.
No cell here re-runs model training -- everything is read from disk.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

TABLES_DIR = Path("../outputs/tables")

PLAYER_SEASON_PATH = TABLES_DIR / "player_season_attribution_v011.json"
SEASON_REPORT_PATH = TABLES_DIR / "season_aggregation_v011_report.json"
STABILITY_REPORT_PATH = TABLES_DIR / "aggregation_stability_v011_report.json"

player_season = (
    pd.read_json(PLAYER_SEASON_PATH) if PLAYER_SEASON_PATH.exists() else None
)
season_report = (
    json.loads(SEASON_REPORT_PATH.read_text()) if SEASON_REPORT_PATH.exists() else None
)
stability_report = (
    json.loads(STABILITY_REPORT_PATH.read_text()) if STABILITY_REPORT_PATH.exists() else None
)

if player_season is not None:
    print(f"Loaded {len(player_season):,} player-season rows")
else:
    print("player_season_attribution_v011.json not found -- run `make run-season-aggregation` first")

## 1. The confidence taxonomy (Phase 1)

Four independent taxonomies -- deliberately never collapsed into one number.
`derive_confidence_tier` is the ONLY derived summary, and it is an ORDINAL
label (high/medium/low/unavailable), never a probability.

In [ ]:
from mlb_luck_score.scoring.component_confidence import (
    DATA_QUALITY_VALUES,
    DOMAIN_VALUES,
    MODEL_STATUS_VALUES,
    SUPPORT_VALUES,
    TIER_VALUES,
)

print("A. Model-status confidence:", MODEL_STATUS_VALUES)
print("B. Row-level data quality:  ", DATA_QUALITY_VALUES)
print("C. Statistical support:    ", SUPPORT_VALUES)
print("D. Domain status:          ", DOMAIN_VALUES)
print("Derived confidence tier:   ", TIER_VALUES)

## 2. Season-level accounting identity (Phase 3)

`season_observed_minus_expected = season_contact_component +
season_unexplained_residual_component + season_defensive_execution_component
+ season_advancement_component`, re-verified AFTER aggregation
(`season_identity_holds_for_every_row` in the Phase 9 report).

In [ ]:
if season_report is not None:
    print("Model selection winners:", season_report["model_selection_winners"])
    print("Component model status:", season_report["component_model_status"])
    print("Season identity holds for every row:", season_report["season_identity_holds_for_every_row"])
    print("Total scored plays:", season_report["total_scored_plays"])
    print("Player-season rows:", season_report["player_season_row_count"])

## 3. Qualification status distribution (Phase 5)

Predetermined thresholds (`primary`/`strict`/`lenient`), never searched for
an appealing leaderboard -- see `mlb_luck_score.scoring.qualification`.

In [ ]:
if season_report is not None:
    print("Threshold set used:", season_report["qualification_threshold_set"])
    print("Qualification counts:", season_report["qualification_counts"])

if player_season is not None:
    display(player_season["qualification_status"].value_counts())

## 4. Qualified leaderboard, with uncertainty (illustrative only)

Sorted by `observed_minus_expected_per_100` among `qualified` rows only --
this is NOT the final public-facing composite score (that is an explicit
NEXT phase, deliberately out of scope for Version 0.11).

In [ ]:
if player_season is not None:
    qualified = player_season[player_season["qualification_status"] == "qualified"].copy()
    display(
        qualified.sort_values("observed_minus_expected_per_100", ascending=False)
        .head(20)[
            [
                "batter",
                "season",
                "eligible_batted_balls",
                "games",
                "observed_minus_expected_per_100",
                "observed_minus_expected_per_100_ci_low",
                "observed_minus_expected_per_100_ci_high",
                "share_of_value_from_provisional_components",
                "component_coverage_fraction",
            ]
        ]
    )

## 5. Bootstrap design and interval width vs. sample size (Phase 4/6)

Sampling-variability intervals ONLY -- conditional on the frozen models, game
_pk-clustered, percentile method (see `aggregation_uncertainty`'s module
docstring for why, chosen before any real result was examined).

In [ ]:
if season_report is not None:
    print(json.dumps(season_report["bootstrap_design"], indent=2))

if stability_report is not None:
    display(pd.DataFrame(stability_report["interval_width_by_sample_size"]))

## 6. Split-half reliability (Phase 6)

Calendar first-half-vs-second-half and odd/even-`game_pk` correlations of
`observed_minus_expected_per_100`, restricted to players with adequate
eligible-play counts in BOTH halves. Descriptive development diagnostics
over 2021-2024 only -- not an untouched validation claim.

In [ ]:
if stability_report is not None:
    print(json.dumps(stability_report["split_half_reliability"], indent=2))

## 7. Qualification-threshold and provisional-pathway sensitivity (Phase 6)

How much does the top-N ranking change across the three predetermined
threshold sets, and when provisional/limited-evidence component value is
excluded entirely? Reports the biggest individual movers, not just an
aggregate correlation.

In [ ]:
if stability_report is not None:
    print(json.dumps(stability_report["qualification_threshold_sensitivity"], indent=2))
    print("Rank correlation (full vs. excluding provisional):",
          stability_report["provisional_exclusion_sensitivity"]["rank_correlation_full_vs_non_provisional"])
    display(pd.DataFrame(
        stability_report["provisional_exclusion_sensitivity"]["top_25_biggest_movers_when_provisional_excluded"]
    ))

## 8. Component covariance at the player-season level (Phase 6)

Descriptive only -- no causal claim about why components co-move.

In [ ]:
if stability_report is not None:
    cov = stability_report["component_covariance"]
    print(f"n_players = {cov['n_players']}")
    if cov["correlation"] is not None:
        display(pd.DataFrame(cov["correlation"]))

## 9. Confirmation: 2025 untouched

`mlb_luck_score.scoring.run_season_aggregation`/`run_attribution_ledger` both
independently re-check every season value in the loaded input file against
`SCRIPT_SEASONS = (2021, 2022, 2023, 2024)` before anything else runs, and
neither script exposes an `--allow-final-evaluation` escape hatch.

In [ ]:
if season_report is not None:
    print(json.dumps(season_report["seasons"], indent=2))